In [ ]:
# Last amended: 23rd Nov, 2026
# My folder:
# GitHUB repo:
#


# 🎓 Lab Tutorial: Understanding & Querying GraphRAG with LlamaIndex and Ollama

Welcome to this hands-on lab on **GraphRAG (Graph Retrieval-Augmented Generation)**!

---

## 📌 What You Will Learn Today:
1. **Why standard Vector RAG fails** on multi-hop relationships and cross-document connections.
2. **How Knowledge Graphs represent information** using `(Subject) ➔ [Predicate] ➔ (Object)` triples.
3. **How LlamaIndex extracts graphs** automatically using local open-weight LLMs via **Ollama**.
4. **How to visualize and interact with the graph**.
5. **How to query the graph** using hybrid graph traversal + vector retrieval.

## 1. Setup & Environment Verification
First, let's import the necessary modules and check our connection to the local **Ollama** server.

In [ ]:
%pip install pyvis

In [1]:
# CRITICAL FOR JUPYTER NOTEBOOKS:
# Jupyter already runs inside an active asyncio event loop.
# nest_asyncio patches the loop so LlamaIndex can run its extraction calls seamlessly.
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
import sys
from pathlib import Path

# Base paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
#STORAGE_DIR = PROJECT_ROOT / "storage"
#OUTPUT_DIR = PROJECT_ROOT / "output"

# Ensure output directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#STORAGE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Inspecting the Sample Dataset
Our dataset revolves around a complex corporate and research technology scenario involving four interconnected documents:
- `aurora_technologies.txt`: Quantum computing hardware firm and key founders.
- `helios_capital.txt`: Deep tech venture firm orchestrating cross-company investments.
- `synapse_dynamics.txt`: Advanced robotics company developing humanoid robots.
- `project_nexus_investigation.txt`: Investigative report revealing the covert integration of quantum chips into autonomous robotics.

Let's preview these documents:

In [3]:
for doc_file in sorted(DATA_DIR.glob('*.txt')):
    print(f"📄 File: {doc_file.name}")
    with open(doc_file, 'r') as f:
        preview = f.read(250).replace('\n', ' ')
        print(f"   Preview: {preview}...\n")

📄 File: aurora_technologies.txt
   Preview: # Company Overview: Aurora Technologies Inc.  Aurora Technologies is a pioneering quantum computing hardware company founded in 2021 by Dr. Elena Vance and Marcus Thorne. Headquartered in Seattle, Washington, Aurora specializes in room-temperature su...

📄 File: helios_capital.txt
   Preview: # Investment Dossier: Helios Capital Management  Helios Capital is a venture capital and private equity firm based in New York City, specializing in deep tech, quantum infrastructure, and autonomous systems. Helios is managing partner for the $1.2B "...

📄 File: project_nexus_investigation.txt
   Preview: # Investigative Report: The Secret Architecture of Project Nexus Source: Tech Sentinel Investigative Desk Classification: Public Domain Investigation Summary  In late 2024, confidential whistleblower memos revealed the existence of "Project Nexus", a...

📄 File: synapse_dynamics.txt
   Preview: # Corporate Profile: Synapse Dynamics  Synapse Dynamics is 

## 3. Configuring LlamaIndex with Local Ollama
We configure LlamaIndex to use:
- **LLM**: `llama3.2` (or `llama3`, `mistral`, `qwen2.5`)
- **Embeddings**: `nomic-embed-text`

In [4]:
from llama_index.core import Settings, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
#from config import OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL, OLLAMA_BASE_URL, OLLAMA_REQUEST_TIMEOUT, DATA_DIR

# 2.0
llm = Ollama(model="qwen2.5:latest",
             request_timeout=800.0,
             temperature = 0.9
            )

# 2.1 Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

# 2.2
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} document sections.")

Loaded 4 document sections.


## 4. Building the PropertyGraphIndex (GraphRAG)
The core step of GraphRAG: the LLM scans each chunk and extracts **Entities** (people, organizations, technologies) and **Relations** between them into a persistent graph store.

In [5]:
# Required for Jupyter: allow nested event loops
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import PropertyGraphIndex
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor

# Extract entity-relation-entity paths
kg_extractor = SimpleLLMPathExtractor(
    llm=llm,
    max_paths_per_chunk=10,
    num_workers=1
)

print("Constructing Property Graph Index via Ollama...")
# Note: use_async=False prevents "RuntimeError: asyncio.run() cannot be called from a running event loop" in Jupyter
index = PropertyGraphIndex.from_documents(
    documents,
    kg_extractors=[kg_extractor],
    use_async=False,
    show_progress=True
)
print("Graph index built successfully!")

Constructing Property Graph Index via Ollama...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-23 15:46:55,643 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-11-23 15:47:20,251 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-23 15:47:45,613 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-23 15:48:02,461 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-11-23 15:48:12,493 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

2026-11-23 15:48:13,787 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Generating embeddings:   0%|          | 0/132 [00:00<?, ?it/s]

2026-11-23 15:48:15,694 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:17,606 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:19,526 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:21,439 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:23,351 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:25,262 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:27,172 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:29,090 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:31,005 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-11-23 15:48:32,923 - INFO - HTTP Request: POST http://localhost:1143

Graph index built successfully!


In [ ]:
# Ensure visualization dependencies are installed if you haven't already:
# !pip install pyvis networkx

# Render the interactive graph directly inside the Jupyter cell output
# index.property_graph_store.show_jupyter_graph()


In [6]:
index.property_graph_store.save_networkx_graph(name="/home/ashok/Documents/graph_rag/graph.html")


In [7]:
from IPython.display import IFrame

# 1. Export the graph to a standalone HTML file using PyVis
index.property_graph_store.save_networkx_graph(name="./graph.html")

# 2. Display the HTML file directly inside your notebook cell
IFrame(src="./graph.html", width="100%", height="800px")


## 6. Querying: Vector RAG vs. GraphRAG Comparison
Now let's ask a multi-hop question that requires bridging information across **3 distinct documents**:

> *"How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"*

In [ ]:
query_text = "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"

# 1. GraphRAG Query Engine
graph_engine = index.as_query_engine(include_text=True, similarity_top_k=3)
response_graph = graph_engine.query(query_text)

print("=== 💡 GRAPHRAG RESPONSE ===")
print(response_graph.response)

## 7. Hands-on Student Exercises

1. **Trace a New Pathway**: Add a new document to `data/` about a government regulatory hearing (`senate_hearing.txt`) and rebuild the graph.
2. **Entity Types**: Modify `SimpleLLMPathExtractor` to specify exact schemas: `schema={'PERSON': ['CO_FOUNDED', 'ADVISED', 'COLLABORATED_WITH']}`.
3. **Compare Retrieval Top-K**: Test how increasing `similarity_top_k` in Vector RAG still struggles with relational queries compared to Graph traversal.